<a href="https://colab.research.google.com/github/funny1vamp/project-for-dl/blob/main/mlp_s3_rain_model_partC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3 — European Rain Forecast: PART C，改进后的模型

在原 PART C notebook 的回放框架上（数据、`make_request`、`check`、`crps`、`evaluate`、`compare` 一行未改），构建并评估一个 **hurdle + LightGBM** 模型，然后生成提交文件。

**相对基线的改进**

| 方面 | 基线（气候态 / 随机森林） | 本 notebook |
|---|---|---|
| 分布形状 | 气候态：不读输入；随机森林：成员是叶子均值，几乎不为 0、散布很窄 | 「干或湿」混合分布：P(湿) × 湿时雨量分布，成员取 100 个分位数（PART A 第 3、9 节） |
| P(湿) | 城市 × 月份的频率 | 每个提前量一个 LightGBM 分类器，在留出期上做 Platt 重校准 |
| 湿时雨量 | 城市 × 月份的经验分位数 | LightGBM 预测 log(雨量) 的中心 + 留出期残差的经验分布 |
| 气候态先验 | 城市 × 月份，单月格子噪声大 | 城市 × 月 × **目标 UTC 小时**，在相邻月份和小时间平滑后作为特征（PART A 第 4、5 节） |
| 近期状态（+6 h） | 随机森林：当前雨量、6 h 雨量、湿度、云量、月份 | 3/6/24/48 h 雨量、湿小时数、距上次降雨时长、云量、湿度、露点差、温度/风的 6–24 h 变化（PART A 第 6、8 节） |
| 空间信息 | 无 | **按当地风向**选取约 450 km 外的上风向城市、固定西风方向的上游城市、300 km 内邻居、约 1400 km 外的远西城市、全面板湿比例（PART A 第 7 节） |
| 日周期 | 无 | 目标时刻的**本地太阳时**（UTC + 经度/15），与 6 h 变化量并列（PART A 第 5、8 节） |
| 训练样本 | 只用 4 个挑战时刻 | 每 3 小时一个发布时刻（包含 4 个挑战时刻），样本量翻倍 |
| 工程 | — | 特征代码只在 `agent.py` 中写一份，训练和线上共用；模型存为 LightGBM 文本 + `.npz`（不依赖 pickle）；任何异常自动退回气候态 |

**运行方式**：从上到下依次运行。第 7 节（2024 验证）和第 8 节（2025–2026 回放）各训练一次，第 9 节用全部数据重新训练并写出提交文件。Colab 免费版上整本大约需要 10–20 分钟。按原 notebook 的建议，调参只看第 7 节，第 8 节每个准备提交的版本只看一次。

---

## 1. Setup and download

`MLARENA_API_KEY` 只在需要下载数据或提交时才用到：优先读 Colab Secrets，其次读环境变量。

In [ ]:
!pip install -q mlarena-sdk lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.1 MB/s eta 0:00:00


In [ ]:
import os

import numpy as np
import pandas as pd
import requests

CHALLENGE_ID = 177


def get_client():
    """ML-Arena client, created only when a download or submit needs it."""
    import mlarena
    key = os.environ.get("MLARENA_API_KEY")
    if key is None:
        try:
            from google.colab import userdata
            key = userdata.get("MLARENA_API_KEY")
        except Exception:
            pass
    if not key:
        raise RuntimeError("Set MLARENA_API_KEY (Colab Secrets or env).")
    return mlarena.connect(api_key=key)

In [ ]:
FILE = "weather_europe_2020_2026.csv.gz"

if not os.path.exists(FILE):
    listing = get_client().datasets(CHALLENGE_ID)
    [meta] = [f for ds in listing["datasets"] for f in ds["files"]
              if f["label"] == FILE]
    resp = requests.get(meta["download_url"], timeout=300)
    resp.raise_for_status()
    with open(FILE, "wb") as fh:
        fh.write(resp.content)
print(FILE, f"{os.path.getsize(FILE) / 1e6:.1f} MB")

weather_europe_2020_2026.csv.gz 28.2 MB


---

## 2. The data, as the challenge sees it

与原 notebook 完全相同：`X` 为 (hours, cities, features)，城市按挑战的 PANEL 顺序排列。

In [ ]:
PANEL = [
    ("Amsterdam", "NL"), ("Athens", "GR"), ("Belgrade", "RS"),
    ("Berlin", "DE"), ("Brussels", "BE"), ("Bucharest", "RO"),
    ("Budapest", "HU"), ("Chisinau", "MD"), ("Copenhagen", "DK"),
    ("Dublin", "IE"), ("Helsinki", "FI"), ("Kyiv", "UA"),
    ("London", "GB"), ("Madrid", "ES"), ("Minsk", "BY"),
    ("Moscow", "RU"), ("Oslo", "NO"), ("Paris", "FR"),
    ("Prague", "CZ"), ("Riga", "LV"), ("Rome", "IT"),
    ("Sarajevo", "BA"), ("Sofia", "BG"), ("Stockholm", "SE"),
    ("Vienna", "AT"), ("Warsaw", "PL"), ("Zagreb", "HR"),
    ("Istanbul", "TR"), ("Saint Petersburg", "RU"), ("Hamburg", "DE"),
    ("Munich", "DE"), ("Frankfurt am Main", "DE"), ("Milan", "IT"),
    ("Naples", "IT"), ("Palermo", "IT"), ("Barcelona", "ES"),
    ("Valencia", "ES"), ("Sevilla", "ES"), ("Marseille", "FR"),
    ("Birmingham", "GB"), ("Glasgow", "GB"), ("Kraków", "PL"),
    ("Göteborg", "SE"), ("Odesa", "UA"), ("Kharkiv", "UA"),
]
FEATURES = ["temperature", "rain", "wind_speed", "wind_direction",
            "humidity", "clouds", "visibility", "snow"]

df = pd.read_csv(FILE, dtype={"city_name": "category",
                              "country_code": "category"})
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True,
                                 format="ISO8601")
key = df["city_name"].astype(str) + "|" + df["country_code"].astype(str)
position = {f"{n}|{c}": j for j, (n, c) in enumerate(PANEL)}
df["pos"] = key.map(position)
assert df["pos"].notna().all() and df["pos"].nunique() == len(PANEL)
df = df.sort_values(["timestamp", "pos"])

hours = pd.DatetimeIndex(df["timestamp"].unique())
assert (hours[1:] - hours[:-1] == pd.Timedelta(hours=1)).all()
X = df[FEATURES].to_numpy(float).reshape(len(hours), len(PANEL),
                                         len(FEATURES))
coords = df.groupby("pos")[["latitude", "longitude"]].first()
rain = X[:, :, FEATURES.index("rain")]          # (hours, cities)
month = hours.month.to_numpy()
del df, key
print(X.shape, hours[0], "->", hours[-1])

(54096, 45, 8) 2020-01-01 00:00:00+00:00 -> 2026-03-03 23:00:00+00:00


In [ ]:
HORIZONS = [6, 48]
ISSUE_HOURS = (5, 11, 17, 23)          # UTC, every 6 hours
SPLIT = hours.searchsorted(pd.Timestamp("2025-01-01", tz="UTC"))


def issue_times(start, end):
    """Issue hours in [start, end) with 48 h of history and their +48 h."""
    i = np.arange(max(start, 47), end - max(HORIZONS))
    return i[np.isin(hours.hour[i], ISSUE_HOURS)]


TRAIN = issue_times(0, SPLIT)
TEST = issue_times(SPLIT, len(hours))
print(len(TRAIN), "training issue times |", len(TEST), "replayed,",
      hours[TEST[0]], "->", hours[TEST[-1]])

7293 training issue times | 1700 replayed, 2025-01-01 05:00:00+00:00 -> 2026-03-01 23:00:00+00:00


---

## 3. The request and the local score

原 notebook 的 `make_request`、`check`、`crps`、`evaluate`、`compare`，原样保留。

In [ ]:
import json


def iso(ts):
    return ts.strftime("%Y-%m-%dT%H:%M:%SZ")


def window(i):
    """The history sent at issue hour i: (cities, 48 h, features)."""
    return np.nan_to_num(X[i - 47:i + 1].transpose(1, 0, 2), nan=0.0)


def make_request(i):
    return {
        "issue_time": iso(hours[i]),
        "timestamps": [iso(t) for t in hours[i - 47:i + 1]],
        "observed": [True] * 48,
        "cities": [{"name": n, "country": c,
                    "lat": float(coords.loc[j, "latitude"]),
                    "lon": float(coords.loc[j, "longitude"])}
                   for j, (n, c) in enumerate(PANEL)],
        "feature_names": FEATURES,
        "history": window(i).tolist(),
        "horizons": HORIZONS,
        "max_members": 100,
    }


def check(response, request):
    """The response as a (cities, horizons, M) array, or raise."""
    json.dumps(response, allow_nan=False)      # JSON, no NaN/inf
    ens = np.asarray(response["rain"], dtype=float)
    shape = (len(request["cities"]), len(request["horizons"]))
    assert ens.ndim == 3 and ens.shape[:2] == shape, ens.shape
    assert 1 <= ens.shape[2] <= request["max_members"], ens.shape
    assert np.isfinite(ens).all()
    return ens


def crps(members, y):
    """CRPS of ensembles (..., M) against outcomes (...), in mm."""
    x = np.sort(np.maximum(members, 0.0), axis=-1)  # clipped at 0
    M = x.shape[-1]
    k = np.arange(1, M + 1)
    spread = (x * (2 * k - M - 1)).sum(axis=-1) / M ** 2
    return np.abs(x - y[..., None]).mean(axis=-1) - spread

In [ ]:
def evaluate(agent, issues=TEST):
    """One row per run: CRPS per horizon (mean over cities), score."""
    rows = []
    for i in issues:
        request = make_request(i)
        ens = check(agent.predict(request), request)
        rows.append({f"CRPS {h}h": crps(ens[:, k], rain[i + h]).mean()
                     for k, h in enumerate(HORIZONS)})
    runs = pd.DataFrame(rows, index=hours[issues])
    runs["score"] = runs.mean(axis=1)
    return runs


def compare(a, b, week=28):
    """Mean of a - b over the same runs, and its weekly-block SE."""
    d = (a["score"] - b["score"]).to_numpy()
    blocks = d[:len(d) // week * week].reshape(-1, week).mean(axis=1)
    return d.mean(), blocks.std(ddof=1) / np.sqrt(len(blocks))

---

## 4. The baseline: city × month climatology

原 notebook 的基线，在 2020–2024 上拟合，作为比较对象。

In [ ]:
M = 100
LEVELS = (np.arange(M) + 0.5) / M


class Climatology:
    """For each city and valid month, M quantiles of the past rain."""

    def __init__(self, end):
        r, m = rain[:end], month[:end]
        self.table = np.stack([np.quantile(r[m == k], LEVELS, axis=0).T
                               for k in range(1, 13)])  # (12, cities, M)

    def predict(self, request):
        issue = pd.Timestamp(request["issue_time"])
        months = [(issue + pd.Timedelta(hours=h)).month
                  for h in request["horizons"]]
        members = self.table[np.array(months) - 1]    # (2, cities, M)
        return {"rain": members.transpose(1, 0, 2).tolist()}


runs_clim = evaluate(Climatology(SPLIT))
print(runs_clim.mean().round(4).to_string())

CRPS 6h     0.0760
CRPS 48h    0.0758
score       0.0759


---

## 5. `agent.py`：特征与模型只写一份

下面这个单元格写出提交用的 `agent.py`。第 6 节的训练代码直接 `import agent`，调用其中的 `base_features` / `horizon_features` 构造训练特征，所以**训练和线上推理用的是同一份代码**。

- `base_features(hist, names, lat, lon)`：从 48 小时历史中提取本地状态、邻居和上风向特征。按**名字**取列；湿度为 0 视为缺失（挑战把缺失值发成 0）。
- `horizon_features(...)`：目标时刻的太阳时、年内时间，以及平滑后的气候态先验。
- `mixture_members`：用 P(湿) 和湿时分位数构造 100 个成员。湿的成员至少 0.1 mm，最多 30 mm，并四舍五入到 0.1 mm（观测值就是 0.1 mm 精度）。
- `Agent()` 不带参数时从自身所在目录加载模型；notebook 里则直接传入刚训练好的模型。`predict` 出任何异常都会退回城市 × 月份气候态。

In [ ]:
%%writefile agent.py
"""European Rain Forecast (challenge 177): hurdle model with LightGBM.

    predict(request) -> {"rain": (45, 2, M) nested list}, M = 100

For each city and lead time the forecast is a "dry or wet" mixture:
  * P(wet): a LightGBM classifier (one per lead time), Platt-recalibrated
    on a held-out period;
  * amount if wet: a LightGBM regressor for the centre of log(rain), plus
    the empirical distribution of its held-out residuals;
  * member k is the quantile at level (k - 0.5) / M of that mixture.

The same file does the feature engineering for training (the notebook
imports it) and live prediction, so the model sees at prediction exactly
what it saw in training. Any failure falls back to a smoothed
city x month climatology instead of crashing the run.
"""
import json
import os
from datetime import datetime, timedelta

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
M = 100
LEVELS = (np.arange(M) + 0.5) / M
HORIZONS = (6, 48)
MAX_MM = 30.0          # the file's maximum is 24.9 mm: never forecast more
WET = 0.1              # a wet hour: at least 0.1 mm

BASE_NAMES = [
    "rain_now", "rain_3h", "rain_6h", "rain_24h", "rain_48h", "rain_max_24h",
    "wet_6h", "wet_48h", "hours_since_wet",
    "clouds_now", "clouds_6h_mean", "clouds_chg_6h",
    "hum_now", "hum_chg_6h", "dewpoint_spread",
    "temp_now", "temp_chg_6h", "temp_chg_24h",
    "wind_now", "wind_chg_6h", "wind_sin", "wind_cos",
    "snow_24h",
    "panel_wet_now", "panel_wet_6h",
    "near_wet_now", "near_rain_6h",
    "upwind_rain_3h", "upwind_wet_6h",
    "west_wet_6h", "far_west_wet_24h",
]
HORIZON_NAMES = [
    "solar_sin", "solar_cos", "doy_sin", "doy_cos",
    "clim_p", "clim_logamt", "lat", "lon", "city",
]
FEATURE_NAMES = BASE_NAMES + HORIZON_NAMES
CATEGORICAL = ["city"]


# --------------------------------------------------------------- geometry
def geometry(lat, lon):
    """Great-circle distance (km) and bearing (deg) from city i to city j."""
    la, lo = np.radians(lat), np.radians(lon)
    dlat = la[None, :] - la[:, None]
    dlon = lo[None, :] - lo[:, None]
    a = (np.sin(dlat / 2) ** 2
         + np.cos(la[:, None]) * np.cos(la[None, :]) * np.sin(dlon / 2) ** 2)
    dist = 2 * 6371.0 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    y = np.sin(dlon) * np.cos(la[None, :])
    x = (np.cos(la[:, None]) * np.sin(la[None, :])
         - np.sin(la[:, None]) * np.cos(la[None, :]) * np.cos(dlon))
    bearing = np.degrees(np.arctan2(y, x)) % 360
    return dist, bearing


def _ring(dist, centre, width):
    """Gaussian weight on distance, 0 on the city itself."""
    w = np.exp(-0.5 * ((dist - centre) / width) ** 2)
    np.fill_diagonal(w, 0.0)
    return w


def _cone(bearing, direction):
    """Weight of source j seen from i in `direction` (deg), cos^2 cone."""
    c = np.cos(np.radians(bearing - direction))
    return np.maximum(c, 0.0) ** 2


def _weighted(w, v, min_total=0.05):
    """sum_j w[..., i, j] v[..., j] / sum_j w, NaN where w is ~empty."""
    total = w.sum(axis=-1)
    out = np.einsum("...ij,...j->...i", w, v) / np.maximum(total, 1e-9)
    return np.where(total > min_total, out, np.nan)


# --------------------------------------------------------------- features
def base_features(hist, names, lat, lon):
    """(N, C, len(BASE_NAMES)) from histories (N, C, 48, F).

    `names` are the feature names of the last axis (request order), so
    the columns are found by name, never by position.
    """
    hist = np.nan_to_num(np.asarray(hist, dtype=np.float64), nan=0.0,
                         posinf=0.0, neginf=0.0)
    f = {n: hist[..., names.index(n)] for n in
         ("rain", "clouds", "humidity", "temperature", "wind_speed",
          "wind_direction", "snow")}
    r = np.maximum(f["rain"], 0.0)
    wet = r >= WET

    rev = wet[..., ::-1]
    since = np.where(rev.any(-1), rev.argmax(-1), 48).astype(float)

    hum = np.where(f["humidity"] > 0, f["humidity"], np.nan)  # 0 = missing
    t = f["temperature"]
    # Dew point (Magnus); temperature minus dew point = how far from saturation.
    with np.errstate(invalid="ignore", divide="ignore"):
        gam = np.log(hum[..., -1] / 100) + 17.62 * t[..., -1] / (243.12 + t[..., -1])
        dew = 243.12 * gam / (17.62 - gam)
    wd = np.radians(f["wind_direction"][..., -1])

    rain_3h = r[..., -3:].sum(-1)
    wet_6h = wet[..., -6:].mean(-1)
    wet_24h = wet[..., -24:].mean(-1)
    rain_6h = r[..., -6:].sum(-1)
    wet_now = wet[..., -1].astype(float)

    # Neighbours. Upwind = the direction the wind comes FROM at the city.
    dist, bearing = geometry(np.asarray(lat, float), np.asarray(lon, float))
    near = (dist < 350).astype(float)
    np.fill_diagonal(near, 0.0)
    ring = _ring(dist, 450.0, 200.0)
    cone = _cone(bearing[None], np.degrees(wd)[..., None])     # (N, C, C)
    up = ring[None] * cone
    west = ring * _cone(bearing, 270.0)
    far_west = _ring(dist, 1400.0, 500.0) * _cone(bearing, 270.0)

    cols = [
        r[..., -1], rain_3h, rain_6h, r[..., -24:].sum(-1), r.sum(-1),
        r[..., -24:].max(-1),
        wet_6h, wet.mean(-1), since,
        f["clouds"][..., -1], f["clouds"][..., -6:].mean(-1),
        f["clouds"][..., -1] - f["clouds"][..., -7],
        hum[..., -1], hum[..., -1] - hum[..., -7], t[..., -1] - dew,
        t[..., -1], t[..., -1] - t[..., -7], t[..., -1] - t[..., -25],
        f["wind_speed"][..., -1],
        f["wind_speed"][..., -1] - f["wind_speed"][..., -7],
        np.sin(wd), np.cos(wd),
        f["snow"][..., -24:].sum(-1),
        np.broadcast_to(wet_now.mean(-1, keepdims=True), wet_now.shape),
        np.broadcast_to(wet_6h.mean(-1, keepdims=True), wet_6h.shape),
        _weighted(near, wet_now), _weighted(near, rain_6h),
        _weighted(up, rain_3h), _weighted(up, wet_6h),
        _weighted(west, wet_6h), _weighted(far_west, wet_24h),
    ]
    return np.stack(cols, axis=-1).astype(np.float32)


def horizon_features(valid_hour, valid_month, valid_doy, lat, lon, city_idx,
                     p_clim, logamt):
    """(N, C, len(HORIZON_NAMES)) for the valid hour of each forecast.

    valid_* are (N,) arrays (UTC); p_clim (C, 12, 24) and logamt (C, 12)
    are already in the same city order as lat / lon / city_idx.
    """
    lon = np.asarray(lon, float)
    n, c = len(valid_hour), len(lon)
    solar = (np.asarray(valid_hour, float)[:, None] + lon[None] / 15.0) % 24
    doy = np.asarray(valid_doy, float)[:, None] * np.ones((1, c))
    m = np.asarray(valid_month)[:, None] - 1
    h = np.asarray(valid_hour)[:, None]
    ci = np.arange(c)[None, :]
    cols = [
        np.sin(2 * np.pi * solar / 24), np.cos(2 * np.pi * solar / 24),
        np.sin(2 * np.pi * doy / 365.25), np.cos(2 * np.pi * doy / 365.25),
        p_clim[ci, m, h], logamt[ci, m],
        np.broadcast_to(np.asarray(lat, float)[None], (n, c)),
        np.broadcast_to(lon[None], (n, c)),
        np.broadcast_to(np.asarray(city_idx, float)[None], (n, c)),
    ]
    return np.stack(cols, axis=-1).astype(np.float32)


# ----------------------------------------------------------- the mixture
def mixture_members(p, mu, grid, resid_q):
    """(n, M) members of P(dry)=1-p at 0, else exp(mu + residual)."""
    p = np.clip(p, 1e-4, 0.999)[:, None]
    u = np.clip((LEVELS[None] - (1 - p)) / p, 0.0, 1.0)
    wet_q = np.exp(mu[:, None] + np.interp(u, grid, resid_q))
    wet_q = np.clip(np.round(wet_q, 1), WET, MAX_MM)   # rain comes in 0.1 mm
    return np.where(LEVELS[None] < 1 - p, 0.0, wet_q)


def platt(p, ab):
    """Recalibrate probabilities: sigmoid(a * logit(p) + b)."""
    p = np.clip(p, 1e-6, 1 - 1e-6)
    z = ab[0] * np.log(p / (1 - p)) + ab[1]
    return 1 / (1 + np.exp(-z))


# ------------------------------------------------------------- artifacts
def save_artifacts(art, folder):
    """Boosters as LightGBM text, arrays in one .npz, the rest as JSON."""
    os.makedirs(folder, exist_ok=True)
    arrays = {k: art[k] for k in ("lat", "lon", "p_clim", "logamt", "clim_q",
                                   "grid")}
    meta = {"panel": art["panel"], "feature_names": FEATURE_NAMES,
            "horizons": {}}
    for h, part in art["horizons"].items():
        for kind in ("clf", "reg"):
            with open(os.path.join(folder, f"{kind}_{h}.txt"), "w") as fh:
                fh.write(part[kind].model_to_string())
        arrays[f"resid_q_{h}"] = part["resid_q"]
        meta["horizons"][str(h)] = {"platt": list(map(float, part["platt"]))}
    np.savez_compressed(os.path.join(folder, "rain_arrays.npz"), **arrays)
    with open(os.path.join(folder, "rain_meta.json"), "w") as fh:
        json.dump(meta, fh)


def load_artifacts(folder):
    import lightgbm as lgb
    with open(os.path.join(folder, "rain_meta.json")) as fh:
        meta = json.load(fh)
    z = np.load(os.path.join(folder, "rain_arrays.npz"))
    art = {k: z[k] for k in ("lat", "lon", "p_clim", "logamt", "clim_q",
                              "grid")}
    art["panel"] = meta["panel"]
    art["horizons"] = {}
    for hs, part in meta["horizons"].items():
        h = int(hs)
        art["horizons"][h] = {
            "clf": lgb.Booster(model_file=os.path.join(folder, f"clf_{h}.txt")),
            "reg": lgb.Booster(model_file=os.path.join(folder, f"reg_{h}.txt")),
            "resid_q": z[f"resid_q_{h}"],
            "platt": np.asarray(part["platt"]),
        }
    return art


def predict_from_features(art, h, X):
    """(n, M) members from a feature matrix (n, len(FEATURE_NAMES))."""
    part = art["horizons"][h]
    p = platt(part["clf"].predict(X), part["platt"])
    mu = part["reg"].predict(X)
    return mixture_members(p, mu, art["grid"], part["resid_q"])


# ----------------------------------------------------------------- agent
def _parse(ts):
    return datetime.fromisoformat(ts.replace("Z", "+00:00"))


class Agent:
    def __init__(self, artifacts=None):
        # The challenge calls Agent() with no argument: load files next to
        # this one. The notebook passes the artifacts it has just fitted.
        self.art = artifacts if artifacts is not None else load_artifacts(HERE)
        self.index = {k: j for j, k in enumerate(self.art["panel"])}

    def _order(self, request):
        """Panel index of each request city, by name and country."""
        return np.array([self.index[f"{c['name']}|{c['country']}"]
                         for c in request["cities"]])

    def _predict(self, request):
        pos = self._order(request)
        lat, lon = self.art["lat"][pos], self.art["lon"][pos]
        hist = np.asarray(request["history"], dtype=float)[None]
        base = base_features(hist, list(request["feature_names"]), lat, lon)
        issue = _parse(request["issue_time"])
        out = []
        for h in request["horizons"]:
            v = issue + timedelta(hours=int(h))
            hor = horizon_features(
                [v.hour], [v.month], [v.timetuple().tm_yday], lat, lon, pos,
                self.art["p_clim"][pos], self.art["logamt"][pos])
            X = np.concatenate([base, hor], axis=-1)[0]
            out.append(predict_from_features(self.art, int(h), X))
        return np.stack(out, axis=1)                       # (C, H, M)

    def _fallback(self, request):
        """Smoothed city x month climatology; pooled for unknown cities."""
        issue = _parse(request["issue_time"])
        q = self.art["clim_q"]                              # (C, 12, M)
        rows = []
        for c in request["cities"]:
            j = self.index.get(f"{c['name']}|{c['country']}")
            table = q[j] if j is not None else np.median(q, axis=0)
            rows.append([table[(issue + timedelta(hours=int(h))).month - 1]
                         for h in request["horizons"]])
        return np.asarray(rows, dtype=float)

    def predict(self, request):
        try:
            members = self._predict(request)
        except Exception:
            try:
                members = self._fallback(request)
            except Exception:
                members = np.zeros((len(request["cities"]),
                                    len(request["horizons"]), M))
        members = np.clip(np.nan_to_num(members, nan=0.0, posinf=MAX_MM),
                          0.0, MAX_MM)
        return {"rain": np.round(members, 2).tolist()}

Writing agent.py


---

## 6. Training tools

- `fit_priors(end)`：只用 `[0, end)` 的数据计算气候态先验，保证不泄漏。
- `train_issues(start, end, step=3)`：每 3 小时一个发布时刻，包含 4 个挑战时刻。
- `build(issues, pri)`：批量构造特征，结果与逐个调用 `window(i)` 完全一致。
- `fit_models(issues, end)`：按时间取最后 15% 的发布时刻做 early stopping、Platt 校准和残差分位数估计，然后用全部行和早停得到的轮数重新训练（`REFIT_FULL`）。
- `fast_eval(art, issues)`：批量版的 `evaluate`，用于快速比较；第 8 节会验证它和官方 `evaluate` 的结果一致。

In [ ]:
import importlib
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression

import agent as A
importlib.reload(A)                    # pick up any edit to agent.py

C = len(PANEL)
LAT = coords["latitude"].to_numpy(float)
LON = coords["longitude"].to_numpy(float)
PANEL_KEYS = [f"{n}|{c}" for n, c in PANEL]
HOUR = hours.hour.to_numpy()
GRID = (np.arange(200) + 0.5) / 200    # levels of the residual quantiles


def smooth(a, axis):
    """Circular [1/4, 1/2, 1/4] smoothing along one axis."""
    return 0.25 * np.roll(a, 1, axis) + 0.5 * a + 0.25 * np.roll(a, -1, axis)


def fit_priors(end):
    """Climatologies from hours [0, end) only, in panel order.

    p_clim  (C, 12, 24): P(wet) by city, month, UTC hour, smoothed over
                         neighbouring months and hours (5 years is thin).
    logamt  (C, 12):     mean log(rain) of wet hours by city and month.
    clim_q  (C, 12, M):  the notebook's city x month quantiles (fallback).
    """
    r, m, hh = rain[:end], month[:end] - 1, HOUR[:end]
    w = r >= A.WET
    num, den = np.zeros((C, 12, 24)), np.zeros((12, 24))
    lsum, lcnt = np.zeros((C, 12)), np.zeros((C, 12))
    logr = np.log(np.where(w, r, 1.0))
    for k in range(12):
        sel_m = m == k
        lsum[:, k] = (logr * w)[sel_m].sum(0)
        lcnt[:, k] = w[sel_m].sum(0)
        for h in range(24):
            sel = sel_m & (hh == h)
            num[:, k, h] = w[sel].sum(0)
            den[k, h] = sel.sum()
    num = smooth(smooth(num, 1), 2)
    den = smooth(smooth(den, 0), 1)
    clim_q = np.stack([np.quantile(r[m == k], LEVELS, axis=0).T
                       for k in range(12)], axis=1)
    return {"p_clim": num / np.maximum(den, 1)[None],
            "logamt": smooth(lsum, 1) / np.maximum(smooth(lcnt, 1), 1),
            "clim_q": clim_q}


def train_issues(start, end, step=3):
    """Like issue_times, but every `step` hours (step divides 6): the four
    challenge hours are always kept, the others add training rows."""
    i = np.arange(max(start, 47), end - max(HORIZONS))
    return i[(HOUR[i] - 5) % step == 0]


def build(issues, pri, chunk=512):
    """Feature matrices and targets, rows = issue-major (issue, city)."""
    Xs, ys = {h: [] for h in HORIZONS}, {h: [] for h in HORIZONS}
    offs = np.arange(-47, 1)
    for s in range(0, len(issues), chunk):
        ii = issues[s:s + chunk]
        # Same values as window(i), for a whole chunk at once.
        hist = np.nan_to_num(X[ii[:, None] + offs].transpose(0, 2, 1, 3))
        base = A.base_features(hist, FEATURES, LAT, LON)
        for h in HORIZONS:
            v = hours[ii + h]
            hor = A.horizon_features(v.hour, v.month, v.dayofyear, LAT, LON,
                                     np.arange(C), pri["p_clim"],
                                     pri["logamt"])
            Xs[h].append(np.concatenate([base, hor], -1)
                         .reshape(-1, len(A.FEATURE_NAMES)))
            ys[h].append(rain[ii + h].reshape(-1))
    return ({h: np.concatenate(Xs[h]) for h in HORIZONS},
            {h: np.concatenate(ys[h]) for h in HORIZONS})


PARAMS_CLF = dict(objective="binary", learning_rate=0.05, num_leaves=63,
                  min_data_in_leaf=300, feature_fraction=0.8,
                  bagging_fraction=0.8, bagging_freq=1, lambda_l2=5.0,
                  verbose=-1, seed=0)
PARAMS_REG = dict(objective="regression", learning_rate=0.05, num_leaves=31,
                  min_data_in_leaf=200, feature_fraction=0.8,
                  bagging_fraction=0.8, bagging_freq=1, lambda_l2=5.0,
                  verbose=-1, seed=0)
MAX_ROUNDS, EARLY_STOP = 3000, 100
REFIT_FULL = True      # refit on all rows with the early-stopped round count


def _dataset(X_, y_, reference=None):
    return lgb.Dataset(X_, y_, feature_name=A.FEATURE_NAMES,
                       categorical_feature=A.CATEGORICAL, reference=reference)


def _train(params, Xa, ya, Xb, yb):
    """Early-stopped on the held-out rows (Xb, yb)."""
    dtr = _dataset(Xa, ya)
    return lgb.train(params, dtr, MAX_ROUNDS,
                     valid_sets=[_dataset(Xb, yb, reference=dtr)],
                     callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])


def _refit(params, booster, X_, y_):
    """All rows, with the early-stopped number of rounds (+10%)."""
    return lgb.train(params, _dataset(X_, y_),
                     max(1, int(booster.best_iteration * 1.1)))


def fit_models(issues, end, holdout=0.15, verbose=True):
    """Priors on [0, end), then per horizon: P(wet) classifier + Platt,
    log-amount regressor + residual quantiles. The last `holdout` share
    of the issue times (in time order) early-stops and calibrates."""
    pri = fit_priors(end)
    Xs, ys = build(issues, pri)
    cut = int(len(issues) * (1 - holdout)) * C          # rows are issue-major
    art = {"panel": PANEL_KEYS, "lat": LAT, "lon": LON, "grid": GRID,
           **pri, "horizons": {}}
    for h in HORIZONS:
        X_, y_ = Xs[h], ys[h]
        wet = (y_ >= A.WET).astype(float)
        Xa, Xb, wa, wb = X_[:cut], X_[cut:], wet[:cut], wet[cut:]

        clf = _train(PARAMS_CLF, Xa, wa, Xb, wb)
        pb = np.clip(clf.predict(Xb), 1e-6, 1 - 1e-6)
        lr = LogisticRegression(C=1e6).fit(np.log(pb / (1 - pb))[:, None], wb)
        ab = np.array([lr.coef_[0, 0], lr.intercept_[0]])

        ia, ib = wa > 0, wb > 0
        la, lb = np.log(y_[:cut][ia]), np.log(y_[cut:][ib])
        reg = _train(PARAMS_REG, Xa[ia], la, Xb[ib], lb)
        resid_q = np.quantile(lb - reg.predict(Xb[ib]), GRID)

        if verbose:
            print(f"+{h:2d} h: {len(X_):,} rows, wet {wet.mean():.3f} | "
                  f"clf {clf.best_iteration} rounds, reg "
                  f"{reg.best_iteration} rounds | Platt a={ab[0]:.2f} "
                  f"b={ab[1]:+.2f}")
        if REFIT_FULL:
            clf = _refit(PARAMS_CLF, clf, X_, wet)
            reg = _refit(PARAMS_REG, reg, X_[wet > 0], np.log(y_[wet > 0]))
        art["horizons"][h] = {"clf": clf, "reg": reg, "resid_q": resid_q,
                              "platt": ab}
    return art


def fast_eval(art, issues):
    """Same rows as evaluate(A.Agent(art), issues), built in batch."""
    Xs, ys = build(issues, art)
    out = {}
    for h in HORIZONS:
        ens = A.predict_from_features(art, h, Xs[h])
        out[f"CRPS {h}h"] = crps(ens, ys[h]).reshape(len(issues), C).mean(1)
    runs = pd.DataFrame(out, index=hours[issues])
    runs["score"] = runs.mean(axis=1)
    return runs


def importance(art, top=12):
    rows = {f"+{h} h": pd.Series(p["clf"].feature_importance("gain"),
                                 index=A.FEATURE_NAMES)
            for h, p in art["horizons"].items()}
    t = pd.DataFrame(rows)
    return (t / t.sum()).sort_values("+6 h", ascending=False).head(top)

---

## 7. 验证：拟合 2020–2023，评估 2024

原 notebook 建议的调参方式。改特征或参数时只看这一节的结果。

In [ ]:
VALID = hours.searchsorted(pd.Timestamp("2024-01-01", tz="UTC"))
VAL = issue_times(VALID, SPLIT)
clim_val = evaluate(Climatology(VALID), VAL)

art_val = fit_models(train_issues(0, VALID), VALID)
runs_val = fast_eval(art_val, VAL)

print(pd.DataFrame({"climatology": clim_val.mean(),
                    "hurdle-lgbm": runs_val.mean()}).T.round(4))
print("hurdle - climatology: %+.4f ± %.4f" % compare(runs_val, clim_val))

+ 6 h: 524,565 rows, wet 0.146 | clf 572 rounds, reg 205 rounds | Platt a=0.99 b=-0.01
+48 h: 524,565 rows, wet 0.146 | clf 106 rounds, reg 44 rounds | Platt a=1.09 b=+0.15
             CRPS 6h  CRPS 48h   score
climatology   0.0811    0.0808  0.0809
hurdle-lgbm   0.0719    0.0799  0.0759
hurdle - climatology: -0.0050 ± 0.0004


In [ ]:
# 各特征在 P(湿) 分类器中的 gain 占比
importance(art_val).round(3)

,+6 h,+48 h
hours_since_wet,0.119,0.016
rain_3h,0.107,0.002
clouds_now,0.097,0.011
city,0.085,0.133
west_wet_6h,0.084,0.100
rain_now,0.055,0.001
clim_p,0.048,0.328
rain_24h,0.032,0.025
temp_now,0.029,0.017
solar_cos,0.025,0.010


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


---

## 8. 回放：拟合 2020–2024，评估 2025-01 → 2026-03

每个准备提交的版本只运行一次。这里用官方 `evaluate`，即逐次构造 request 并调用 `Agent.predict`，与挑战的流程完全一致。同时检查它与批量版 `fast_eval` 的结果是否相同，以此确认训练和推理之间没有不一致。

In [ ]:
art = fit_models(train_issues(0, SPLIT), SPLIT)

runs_mine = evaluate(A.Agent(art), TEST)
runs_fast = fast_eval(art, TEST)
gap = (runs_mine["score"] - runs_fast["score"]).abs().max()
print(f"evaluate vs fast_eval, max |diff| per run: {gap:.2e}")

print(pd.DataFrame({"climatology": runs_clim.mean(),
                    "hurdle-lgbm": runs_mine.mean()}).T.round(4))
print("hurdle - climatology: %+.4f ± %.4f" % compare(runs_mine, runs_clim))
print("参考：starter 0.0746，+6h 条件化 + 48h 气候态 0.0742")

+ 6 h: 656,325 rows, wet 0.146 | clf 615 rounds, reg 197 rounds | Platt a=0.99 b=-0.07
+48 h: 656,325 rows, wet 0.146 | clf 95 rounds, reg 76 rounds | Platt a=1.01 b=-0.08
evaluate vs fast_eval, max |diff| per run: 0.00e+00
             CRPS 6h  CRPS 48h   score
climatology   0.0760    0.0758  0.0759
hurdle-lgbm   0.0669    0.0746  0.0708
hurdle - climatology: -0.0051 ± 0.0003
参考：starter 0.0746，+6h 条件化 + 48h 气候态 0.0742


In [ ]:
# 按月看改进是否稳定（负数 = 优于气候态）
d = (runs_mine["score"] - runs_clim["score"])
d.groupby(d.index.tz_localize(None).to_period("M")).mean().round(4).to_frame("hurdle - clim")

,hurdle - clim
2025-01,-0.0046
2025-02,-0.0038
2025-03,-0.0060
2025-04,-0.0040
2025-05,-0.0045
2025-06,-0.0039
2025-07,-0.0050
2025-08,-0.0033
2025-09,-0.0043
2025-10,-0.0087


---

## 9. 用全部数据重新训练，写出提交文件

线上预测的是未来，所以最终模型使用 2020-01 至 2026-03 的全部数据。模型文件写在 `agent.py` 旁边，之后在一个**临时的空目录**里模拟部署：只拷贝要上传的文件，用 `Agent()`（不带参数）加载并预测一次，同时计时。

In [ ]:
art_all = fit_models(train_issues(0, len(hours)), len(hours))
A.save_artifacts(art_all, ".")

SUBMIT_FILES = (["agent.py", "rain_arrays.npz", "rain_meta.json"]
                + [f"{k}_{h}.txt" for h in HORIZONS for k in ("clf", "reg")])
for f in SUBMIT_FILES:
    print(f"{f:18s} {os.path.getsize(f) / 1e6:6.2f} MB")

+ 6 h: 810,045 rows, wet 0.144 | clf 706 rounds, reg 211 rounds | Platt a=0.96 b=-0.00
+48 h: 810,045 rows, wet 0.144 | clf 106 rounds, reg 112 rounds | Platt a=1.00 b=-0.01
agent.py             0.01 MB
rain_arrays.npz      0.06 MB
rain_meta.json       0.00 MB
clf_6.txt            5.50 MB
reg_6.txt            0.71 MB
clf_48.txt           0.83 MB
reg_48.txt           0.38 MB


In [ ]:
import shutil
import subprocess
import sys
import tempfile
import time

# Deployment test: only the uploaded files, Agent() with no argument.
request = make_request(TEST[-1])
with tempfile.TemporaryDirectory() as tmp:
    for f in SUBMIT_FILES:
        shutil.copy(f, tmp)
    with open(os.path.join(tmp, "request.json"), "w") as fh:
        json.dump(request, fh)
    script = (
        "import json, time, numpy as np\n"
        "t0 = time.time()\n"
        "from agent import Agent\n"
        "a = Agent()\n"
        "req = json.load(open('request.json'))\n"
        "t1 = time.time()\n"
        "out = a.predict(req)\n"
        "t2 = time.time()\n"
        "print(np.asarray(out['rain']).shape, "
        "f'load {t1 - t0:.2f}s, predict {t2 - t1:.2f}s')\n"
        "model = np.round(np.clip(a._predict(req), 0, 30), 2)\n"
        "print('model path used (no fallback):', "
        "np.allclose(out['rain'], model))\n")
    res = subprocess.run([sys.executable, "-c", script], cwd=tmp,
                         capture_output=True, text=True)
    print(res.stdout or res.stderr)

(45, 2, 100) load 3.38s, predict 0.02s
model path used (no fallback): True



将 `SUBMIT = True` 后运行下面的单元格即可提交。**每运行一次就会新建一个提交。** 运行时为 182（带 LightGBM）。第一个分数大约在部署 48 小时后出现。

In [ ]:
SUBMIT = True

if SUBMIT:
    client = get_client()
    result = client.submit(CHALLENGE_ID, files=SUBMIT_FILES,
                           submission_name="hurdle-lgbm-upwind",
                           runtime_id=182)
    submission_id = result["submission_id"]
    print(submission_id)

8986


In [ ]:
# status = get_client().status(submission_id, CHALLENGE_ID)
# print(status["status"], "|", status["last_status_message"])

---

## 10. 可以继续尝试的方向（只在第 7 节上比较）

- **`REFIT_FULL=False`**：不重新训练，Platt 参数和残差分位数与模型完全匹配，但少用最后 15% 的数据。
- **`train_issues(..., step=1)`**：每小时都作为发布时刻，样本量是 step=3 的 3 倍，训练更慢。
- **按季节估计残差分位数**：夏季对流的雨量尾部更重（PART A 第 4 节）。
- **+48 h 与气候态混合**：如果第 7 节显示 +48 h 没有稳定优于气候态，可以把 `p` 和 `mu` 向 `clim_p` / `clim_logamt` 收缩。
- **上风向参数**：`agent.py` 中 `_ring(dist, 450, 200)` 的中心距离和宽度，对应 PART A 第 7 节中大约 6 小时的移动距离。